# Notebook 04: ConvNeXt-Nano ABMIL Training

## 1. Project Setup & Environment Checks
Import required dependencies (PyTorch, timm, Scikit-Learn) and inspect hardware acceleration status.

In [1]:
import os
import json
import time
import numpy as np
import gc
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import timm

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"timm    : {timm.__version__}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

PyTorch : 2.10.0+cu128
CUDA    : True
timm    : 1.0.26
GPU     : Tesla T4


### 1.1 Input Path & Directory Setup
Configure file paths for input feature arrays, target labels, bag IDs and output directories.

In [2]:
NB02 = Path("/kaggle/input/notebooks/mfjmrizvi/02-mil-patch-extraction")
OUT  = Path("/kaggle/working")

X_TRAIN_PATH      = NB02 / "X_train_patches.npy"
Y_TRAIN_PATH      = NB02 / "y_train_labels.npy"
BAG_IDS_TRAIN_PATH= NB02 / "bag_ids_train.npy"
CLASS_WEIGHTS_PATH= NB02 / "class_weights.json"

### 1.2 Hyperparameters & Random Seed Setup
Initialise global training variables, stage 1 batch size, total epoch limit, patience and random seeds.

In [3]:
MODEL_NAME = "convnext_nano"
PATCH_SIZE = 224
SEED       = 42

BS_STAGE1  = 128
EPOCHS_S1  = 15
PATIENCE_S1= 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
torch.manual_seed(SEED)
np.random.seed(SEED)

Device: cuda


### 1.3 Module Imports
Import custom dataset classes, backbone constructors and training orchestration helpers from the shared ABMIL library.

In [4]:
import sys
sys.path.append('/kaggle/input/datasets/mfjmrizvi/cbis-ddsm-project-config')
from abmil_common import (
    build_backbone, PatchClassifier, get_normalisation_tensors, train_stage1_fold)

### 1.4 Data Array Verification
Load and verify dataset shapes, label distributions, cross-validation fold indexes and class weight configurations.

In [5]:
print("Loading arrays...")
X_train_all  = np.load(X_TRAIN_PATH)
y_train_all  = np.load(Y_TRAIN_PATH)
bag_ids_all  = np.load(BAG_IDS_TRAIN_PATH)

fold_ids = np.load(NB02 / "fold_ids.npy")

with open(CLASS_WEIGHTS_PATH) as f:
    raw_cw = json.load(f)
class_weight_dict = {int(k): float(v) for k, v in raw_cw.items()}

print(f"X_train_all : {X_train_all.shape}  dtype={X_train_all.dtype}")
print(f"y_train_all : unique={np.unique(y_train_all)}  "
      f"benign={(y_train_all==0).sum()}  "
      f"malignant={(y_train_all==1).sum()}")
print(f"fold_ids : {fold_ids.shape}  unique folds={sorted(set(fold_ids))}")
print(f"Class weights   : {class_weight_dict}")

Loading arrays...
X_train_all : (58820, 224, 224, 1)  dtype=float32
y_train_all : unique=[0 1]  benign=29225  malignant=29595
fold_ids : (1226,)  unique folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Class weights   : {0: 1.0063301967493585, 1: 0.9937489440783916}


## 2. Hyperparameter Loading & Holdout Logic
Retrieve optimised hyperparameters and set up fold splitting utilities to prevent data leakage.

### 2.1 Load Optuna Optimal Parameters
Import optimal stage 1 hyperparameters (learning rate, weight decay, optimiser) produced by Optuna study runs.

In [6]:
STAGE1_PARAMETER_PATH = Path("/kaggle/input/notebooks/mfjmrizvi/2-6-optuna-run-all/convnext_nano_stage1_optuna_study.json")
with open(STAGE1_PARAMETER_PATH) as f:
    parameter = json.load(f)["best_params"]

OPTIMISER_NAME = parameter["optimiser"]
LR             = parameter["lr"]
WEIGHT_DECAY   = parameter["weight_decay"]
print(f"Using parameters: {parameter}")
all_bags = np.unique(bag_ids_all)

Using parameters: {'optimiser': 'Adam', 'lr': 2.0514306954965613e-05, 'weight_decay': 5.292469708762985e-06}


### 2.2 Memory-Efficient Fold Splitter
Define helper logic to map cross-validation fold indexes directly to corresponding bag and patch locations.

In [7]:
def get_fold_split(fold, all_bags, fold_ids, bag_ids_all, model_name, out_dir):
    if fold == "full":
        rng = np.random.default_rng(999)
        shuffled = rng.permutation(all_bags)
        n_val = int(0.1 * len(shuffled))
        val_bag_indices   = shuffled[:n_val]
        train_bag_indices = shuffled[n_val:]
        save_path = out_dir / f"{model_name}_stage1_full.pth"
    else:
        train_bag_indices = all_bags[fold_ids[all_bags] != fold]
        val_bag_indices   = all_bags[fold_ids[all_bags] == fold]
        save_path = out_dir / f"{model_name}_stage1_fold{fold}.pth"

    train_idx = np.where(np.isin(bag_ids_all, train_bag_indices))[0]
    val_idx   = np.where(np.isin(bag_ids_all, val_bag_indices))[0]

    return train_idx, val_idx, save_path

## 3. Stage 1 Cross-Validation Training Loop
Execute Stage 1 patch-level training across all 5 cross-validation folds plus the full production set.

In [8]:
fold_histories = {}

for fold in [0, 1, 2, 3, 4, "full"]:
    print(f"\n{'='*60}\nTraining Stage 1: {MODEL_NAME} fold={fold}\n{'='*60}")

    train_idx, val_idx, save_path = get_fold_split(
        fold, all_bags, fold_ids, bag_ids_all, MODEL_NAME, OUT
    )
    print(f"Train patches: {len(train_idx)}  Val patches: {len(val_idx)}")

    history, best_val_loss = train_stage1_fold(
        model_name=MODEL_NAME,
        X_all=X_train_all, y_all=y_train_all,
        train_idx=train_idx, val_idx=val_idx,
        class_weight_dict=class_weight_dict,
        optimiser_name=OPTIMISER_NAME, lr=LR, weight_decay=WEIGHT_DECAY,
        save_path=save_path, device=DEVICE,
        epochs=EPOCHS_S1, patience=PATIENCE_S1
    )

    fold_histories[str(fold)] = {
        "best_val_loss": best_val_loss,
        "parameters": parameter,
        "epochs_budget": EPOCHS_S1,
        "patience": PATIENCE_S1,
        "train_patches": len(train_idx),
        "val_patches": len(val_idx),
        "history": history,
    }

    gc.collect(); torch.cuda.empty_cache()


Training Stage 1: convnext_nano fold=0
Train patches: 47252  Val patches: 11568


model.safetensors:   0%|          | 0.00/62.4M [00:00<?, ?B/s]

  Ep 01/15  train=0.6286  val=0.6351  auc=0.6803
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 02/15  train=0.5738  val=0.6329  auc=0.6962
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 03/15  train=0.5405  val=0.6291  auc=0.7073
    ✓ Saved: convnext_nano_stage1_fold0.pth
  Ep 04/15  train=0.5022  val=0.6659  auc=0.6883
  Ep 05/15  train=0.4492  val=0.7281  auc=0.7057
  Ep 06/15  train=0.3859  val=0.7986  auc=0.6805
  Ep 07/15  train=0.2924  val=0.8968  auc=0.6793
  Ep 08/15  train=0.1547  val=1.1651  auc=0.6701
  Early stopping at epoch 8

Training Stage 1: convnext_nano fold=1
Train patches: 47355  Val patches: 11465
  Ep 01/15  train=0.6266  val=0.6784  auc=0.6814
    ✓ Saved: convnext_nano_stage1_fold1.pth
  Ep 02/15  train=0.5522  val=0.6824  auc=0.6592
  Ep 03/15  train=0.5108  val=0.7136  auc=0.6476
  Ep 04/15  train=0.4597  val=0.7657  auc=0.6333
  Ep 05/15  train=0.3886  val=0.8724  auc=0.6279
  Ep 06/15  train=0.2488  val=0.9962  auc=0.6512
  Early stopping at epoch 6



## 4. Stage 1 Summary & Model Checkpoint Persistence
Save fold histories, loss trajectories and training evaluation summaries to disk.

In [9]:
with open(OUT / f"{MODEL_NAME}_stage1_all_folds_summary.json", "w") as f:
    json.dump(fold_histories, f, indent=2)

print("\nAll 6 Stage 1 backbones trained (5 CV folds + 1 production).")


All 6 Stage 1 backbones trained (5 CV folds + 1 production).
